In [ ]:
import polars
import altair

In [ ]:
df = polars.read_parquet("../.extract/s3_metadata")
display(df)
display(round(df["size"].sum() / 1024**4, 2))

In [ ]:
df["size"].sum()
df["key"].str.ends_with(".nc").sum()

In [ ]:
collection_df = (
    df.group_by(
        polars.col("collection"),
    )
    .agg(
        polars.col("size").sum(),
        (polars.col("size").sum() / (1024**2)).alias("size_mb"),
        (polars.col("size").sum() / (1024**3)).alias("size_gb"),
        (polars.col("size").sum() / (1024**4)).alias("size_tb"),
        polars.len().alias("n_files"),
    )
    .with_columns(
        (polars.col("size_mb") / polars.col("n_files")).alias("average_file_size")
    )
    .filter(
        polars.col("size").ne(0),
    )
)

In [ ]:
color = altair.Color(
    shorthand="average_file_size",
    type="quantitative",
    title="Avg File Size (mb)",
    scale=altair.Scale(
        domain=(0, collection_df["average_file_size"].max()), scheme="viridis"
    ),
)
collection_size = (
    altair.Chart(data=collection_df, title=altair.Title("Size of files per collection"))
    .mark_bar()
    .encode(
        x=altair.X("collection", axis=altair.Axis(title="Collection")),
        y=altair.Y("size_tb", axis=altair.Axis(title="Size (TB)")),
        color=color,
    )
)
collection_file_numerosity = (
    altair.Chart(
        data=collection_df,
        title=altair.Title("Number of files per collection"),
    )
    .mark_bar()
    .encode(
        x=altair.X("collection"),
        y=altair.Y("n_files"),
        color=color,
    )
)
average_file_size = (
    altair.Chart(
        data=collection_df,
        title=altair.Title("Average file size per collection"),
    )
    .mark_bar()
    .encode(
        x=altair.X("collection"),
        y=altair.Y("average_file_size", axis=altair.Axis(title="Size (MB)")),
        color=color,
    )
)
# collection_average_file_size =
collection_size | collection_file_numerosity | average_file_size

In [ ]:
binned_df = (
    df.filter(polars.col("size").gt(0))
    .with_columns((polars.col("size") / (1024)).alias("size_kb"))
    .with_columns(
        # Option A: cut() gives evenly distributed MB bucket intervals (e.g., 0-10MB, 10-20MB)
        # Option B: qcut(quantiles=[...]) handles heavily skewed datasets by creating adaptive bins
        polars.col("size_kb").qcut(quantiles=50).alias("size_bin")
    )
    # Group by collection AND the bucket to compress millions of rows into a quick summary table
    .group_by("collection", "size_bin")
    .agg(polars.len().alias("file_count"))
    # Sort them nicely so the buckets sequence correctly in the chart
    .sort("size_bin")
)

file_size_histogram = (
    altair.Chart(binned_df)
    .mark_bar()
    .encode(
        x=altair.X(
            "size_bin:N",  # Ordinal/Nominal because Polars cut returns intervals as string categories
            title="File Size Ranges (KB)",
            axis=altair.Axis(labelAngle=-45),
        ),
        y=altair.Y(
            "file_count:Q",
            title="Number of Files",
        ),
        color=altair.Color(
            "collection:N", legend=None, scale=altair.Scale(scheme="category10")
        ),
    )
    .properties(width=220, height=150, title="File Size Distribution")
    .facet(
        facet=altair.Facet("collection:N", title="Collection File Size Distributions"),
        columns=3,
    )
    .resolve_scale(
        y="independent"  # Allows small collections to remain visible alongside giants
    )
)

file_size_histogram

In [ ]:
import polars as pl
import altair as alt

# 1. Prepare and bucket your data on the Polars side
binned_df = (
    df.filter(pl.col("size").gt(0))
    .with_columns((pl.col("size") / (1024**2)).alias("size_mb"))
    .with_columns(
        # Option A: cut() gives evenly distributed MB bucket intervals (e.g., 0-10MB, 10-20MB)
        # Option B: qcut(quantiles=[...]) handles heavily skewed datasets by creating adaptive bins
        pl.col("size_mb")
        .cut(breaks=[1, 10, 50, 100, 500, 1000, 5000], left_closed=True)
        .alias("size_bin")
    )
    # Group by collection AND the bucket to compress millions of rows into a quick summary table
    .group_by("collection", "size_bin")
    .agg(pl.len().alias("file_count"))
    # Sort them nicely so the buckets sequence correctly in the chart
    .sort("size_bin")
)

# 2. Build the chart using the tiny, compressed table
file_size_histogram = (
    alt.Chart(binned_df)
    .mark_bar()
    .encode(
        x=alt.X(
            "size_bin:N",  # Ordinal/Nominal because Polars cut returns intervals as string categories
            title="File Size Ranges (MB)",
            axis=alt.Axis(labelAngle=-45),
        ),
        y=alt.Y(
            "file_count:Q",
            title="Number of Files",
        ),
        color=alt.Color(
            "collection:N", legend=None, scale=alt.Scale(scheme="category10")
        ),
    )
    .properties(width=220, height=150, title="File Size Distribution")
    .facet(
        facet=alt.Facet("collection:N", title="Collection Distribution Breakdowns"),
        columns=3,
    )
    .resolve_scale(
        y="independent"  # Allows small collections to remain visible alongside giants
    )
)

file_size_histogram

In [ ]:
collection_df = (
    df.group_by(
        polars.col("collection"),
    )
    .agg(
        polars.col("size").sum(),
        polars.len().alias("n_files"),
    )
    .filter(
        polars.col("size").ne(0),
    )
)
altair.Chart(
    data=collection_df, title=altair.Title("Number of files per collection")
).mark_bar().encode(x=altair.X("collection"), y=altair.Y("n_files"))

In [ ]:
import altair as alt

alt.Chart(collection_df).mark_arc(innerRadius=50).encode(
    theta=alt.Theta("size:Q"),
    color=alt.Color("collection:N"),
    tooltip=["collection", "size", "n_files"],
)

In [ ]:
display(lf.head().collect())
print(f"Total files: `{lf.select(polars.len()).collect().item()}`")

In [ ]:
lf.select(polars.col("size").cast(polars.Int128())).collect()["size"].sum()

In [ ]:
collection_df = (
    lf.group_by(
        polars.col("collection"),
    )
    .agg(
        polars.col("size").cast(polars.Int128()).sum().alias("total_size"),
        polars.len().alias("n_files"),
    )
    .collect()
)

In [ ]:
display(collection_df)

In [ ]:
52176371731604
55979016121143